# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
from IPython.display import Markdown, display

display(Markdown("""
# Baseline Rule

## Rule

A page should receive a higher action score if:

- It has a high number of Google Search impressions.
- It has relatively few clicks.
- Its average search position is good.
- It has meaningful traffic from GA4.

This suggests that the page is visible in search but is underperforming and is a good candidate for optimisation.

---

## Reason Codes

| Reason Code | Meaning |
|--------------|---------|
| LOW_CTR_HIGH_IMPRESSIONS | Many impressions but comparatively few clicks |
| HIGH_IMPRESSIONS | Strong search visibility |
| GOOD_POSITION | Already ranks reasonably well |
| LOW_ENGAGEMENT | Users are not engaging with the page |

---

## Action Label

**Optimize Existing Content**

The objective is to improve titles, meta descriptions and on-page content so that existing search visibility converts into more clicks.
"""))


# Baseline Rule

## Rule

A page should receive a higher action score if:

- It has a high number of Google Search impressions.
- It has relatively few clicks.
- Its average search position is good.
- It has meaningful traffic from GA4.

This suggests that the page is visible in search but is underperforming and is a good candidate for optimisation.

---

## Reason Codes

| Reason Code | Meaning |
|--------------|---------|
| LOW_CTR_HIGH_IMPRESSIONS | Many impressions but comparatively few clicks |
| HIGH_IMPRESSIONS | Strong search visibility |
| GOOD_POSITION | Already ranks reasonably well |
| LOW_ENGAGEMENT | Users are not engaging with the page |

---

## Action Label

**Optimize Existing Content**

The objective is to improve titles, meta descriptions and on-page content so that existing search visibility converts into more clicks.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os
import duckdb
from huggingface_hub import hf_hub_download

# Download the sample dataset
parquet_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance_sample.parquet"
)

con = duckdb.connect()

# Build a simple baseline action score
baseline_df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions,

    (
        COALESCE(gsc_impressions,0) * 0.40 +
        COALESCE(gsc_clicks,0) * 0.20 +
        COALESCE(ga4_sessions,0) * 0.20 +
        COALESCE(ga4_engaged_sessions,0) * 0.10 +
        CASE
            WHEN gsc_avg_position BETWEEN 1 AND 10 THEN 10
            WHEN gsc_avg_position BETWEEN 11 AND 20 THEN 5
            ELSE 0
        END
    ) AS action_score,

    CASE
        WHEN gsc_clicks = 0 AND gsc_impressions > 100 THEN 'LOW_CTR_HIGH_IMPRESSIONS'
        WHEN gsc_avg_position <= 10 THEN 'GOOD_POSITION'
        ELSE 'GENERAL_OPTIMIZATION'
    END AS reason_code,

    'Optimize Existing Content' AS action_label

FROM read_parquet('{parquet_file}')
WHERE gsc_data_available IS TRUE
ORDER BY action_score DESC
LIMIT 500
""").df()

# Create output directory if needed
os.makedirs("../outputs", exist_ok=True)

output_path = "../outputs/baseline_action_score.csv"
baseline_df.to_csv(output_path, index=False)

print(f"CSV written to: {output_path}")
print(f"Rows written: {len(baseline_df)}")

baseline_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CSV written to: ../outputs/baseline_action_score.csv
Rows written: 500


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,action_score,reason_code,action_label
0,2026-06-11,client_e547b89c05043229,content_963de14b1f58978f,245826,890,6.326149,685,5,98655.9,GOOD_POSITION,Optimize Existing Content
1,2026-06-29,client_e547b89c05043229,content_eadb33b5df496f4a,49373,173,2.627995,96,4,19813.4,GOOD_POSITION,Optimize Existing Content
2,2026-06-30,client_e547b89c05043229,content_eadb33b5df496f4a,48990,189,2.590467,99,3,19663.9,GOOD_POSITION,Optimize Existing Content
3,2026-06-26,client_e547b89c05043229,content_545bb6cc7081ded3,48953,114,2.586277,52,3,19624.7,GOOD_POSITION,Optimize Existing Content
4,2026-06-12,client_e547b89c05043229,content_963de14b1f58978f,46220,81,6.856274,153,0,18544.8,GOOD_POSITION,Optimize Existing Content
5,2026-06-30,client_06d356715a8ff3b6,content_f88878f155e4838d,45890,329,5.828089,413,0,18514.4,GOOD_POSITION,Optimize Existing Content
6,2026-06-25,client_e547b89c05043229,content_545bb6cc7081ded3,42474,107,2.605735,57,4,17032.8,GOOD_POSITION,Optimize Existing Content
7,2026-06-27,client_e547b89c05043229,content_545bb6cc7081ded3,41051,129,2.605150,60,1,16468.3,GOOD_POSITION,Optimize Existing Content
8,2026-06-29,client_06d356715a8ff3b6,content_f88878f155e4838d,35560,313,5.676069,377,0,14372.0,GOOD_POSITION,Optimize Existing Content
9,2026-06-24,client_e547b89c05043229,content_0ec99ef7d7e11565,30645,50,5.019710,43,0,12286.6,GOOD_POSITION,Optimize Existing Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
import pandas as pd
from IPython.display import display, Markdown

display(Markdown("# Top-20 Review"))

top20 = baseline_df.head(20).copy()

reviews = []

for rank, (_, row) in enumerate(top20.iterrows(), start=1):
    reviews.append({
        "Rank": rank,
        "Action": row["action_label"],
        "Reason": row["reason_code"],
        "Why it's here":
            f"High Action Score ({row['action_score']:.1f}) driven by "
            f"{int(row['gsc_impressions'])} impressions, "
            f"{int(row['gsc_clicks'])} clicks and "
            f"average position {row['gsc_avg_position']:.2f}.",
        "What would make it wrong":
            "Seasonal traffic, temporary ranking changes, or business priorities not captured by the rule."
    })

review_df = pd.DataFrame(reviews)

display(review_df)

# Top-20 Review

,Rank,Action,Reason,Why it's here,What would make it wrong
0,1,Optimize Existing Content,GOOD_POSITION,High Action Score (98655.9) driven by 245826 i...,"Seasonal traffic, temporary ranking changes, o..."
1,2,Optimize Existing Content,GOOD_POSITION,High Action Score (19813.4) driven by 49373 im...,"Seasonal traffic, temporary ranking changes, o..."
2,3,Optimize Existing Content,GOOD_POSITION,High Action Score (19663.9) driven by 48990 im...,"Seasonal traffic, temporary ranking changes, o..."
3,4,Optimize Existing Content,GOOD_POSITION,High Action Score (19624.7) driven by 48953 im...,"Seasonal traffic, temporary ranking changes, o..."
4,5,Optimize Existing Content,GOOD_POSITION,High Action Score (18544.8) driven by 46220 im...,"Seasonal traffic, temporary ranking changes, o..."
5,6,Optimize Existing Content,GOOD_POSITION,High Action Score (18514.4) driven by 45890 im...,"Seasonal traffic, temporary ranking changes, o..."
6,7,Optimize Existing Content,GOOD_POSITION,High Action Score (17032.8) driven by 42474 im...,"Seasonal traffic, temporary ranking changes, o..."
7,8,Optimize Existing Content,GOOD_POSITION,High Action Score (16468.3) driven by 41051 im...,"Seasonal traffic, temporary ranking changes, o..."
8,9,Optimize Existing Content,GOOD_POSITION,High Action Score (14372.0) driven by 35560 im...,"Seasonal traffic, temporary ranking changes, o..."
9,10,Optimize Existing Content,GOOD_POSITION,High Action Score (12286.6) driven by 30645 im...,"Seasonal traffic, temporary ranking changes, o..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
import pandas as pd
from IPython.display import display, Markdown

display(Markdown("# Weak Picks + Leakage Check"))

# Bottom 10 recommendations
weak_picks = baseline_df.sort_values("action_score").head(10).copy()

display(Markdown("## Weak Picks"))
display(weak_picks)

# Leakage check
display(Markdown("## Leakage Check"))

leaky_features = [
    "future_clicks",
    "future_impressions",
    "future_ctr",
    "future_engagement",
    "label",
    "target"
]

display(Markdown("""
The baseline rule was built **only using historical information** available at the decision time.

### Features Used
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

### Features NOT Used
- Future clicks
- Future impressions
- Future CTR
- Any target/label-derived columns

**Conclusion:** No future-window or label-derived inputs were used while constructing the baseline score.
"""))

print("Leakage Check Passed ✅")

# Weak Picks + Leakage Check

## Weak Picks

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,action_score,reason_code,action_label
499,2026-06-04,client_73cda7b4e4f265ea,content_471d9cabce329a66,6917,10,6.529999,14,0,2781.6,GOOD_POSITION,Optimize Existing Content
498,2026-06-18,client_62f4a7e64f5e0096,content_acbcc847f8996314,6930,6,5.448918,<NA>,<NA>,2783.2,GOOD_POSITION,Optimize Existing Content
497,2026-06-29,client_a80fca3f171ed1de,content_9540d884af3e41fd,6945,7,7.886105,0,0,2789.4,GOOD_POSITION,Optimize Existing Content
496,2026-06-25,client_86ebc2f12c01f586,content_6c64fe9b8a2d145d,6866,113,3.197641,71,0,2793.2,GOOD_POSITION,Optimize Existing Content
495,2026-06-03,client_73cda7b4e4f265ea,content_62770e1299963fe4,6949,9,4.518780,13,0,2794.0,GOOD_POSITION,Optimize Existing Content
494,2026-06-02,client_73cda7b4e4f265ea,content_f43118e089ecc69a,6957,3,6.746155,4,0,2794.2,GOOD_POSITION,Optimize Existing Content
493,2026-06-21,client_a80fca3f171ed1de,content_9540d884af3e41fd,6961,3,8.832783,3,0,2795.6,GOOD_POSITION,Optimize Existing Content
492,2026-06-17,client_b77d0d5f08f05e64,content_ac1ddc0c0e79289f,6909,117,5.392821,0,0,2797.0,GOOD_POSITION,Optimize Existing Content
491,2026-06-03,client_8ddc46da5414ffd8,content_7471467133493ce6,6973,1,2.572207,<NA>,<NA>,2799.4,GOOD_POSITION,Optimize Existing Content
490,2026-06-18,client_e547b89c05043229,content_9ef3d7516483e665,6983,25,2.254762,20,3,2812.5,GOOD_POSITION,Optimize Existing Content


## Leakage Check


The baseline rule was built **only using historical information** available at the decision time.

### Features Used
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

### Features NOT Used
- Future clicks
- Future impressions
- Future CTR
- Any target/label-derived columns

**Conclusion:** No future-window or label-derived inputs were used while constructing the baseline score.


Leakage Check Passed ✅


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.